In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, Dense, Dropout, Concatenate, Bidirectional
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [ ]:
# 1. Check GPU availability

print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
print("TensorFlow version:", tf.__version__)
print("GPU Device:", tf.test.gpu_device_name())

# 2. Configure GPU memory growth (prevents memory errors)
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print('GPU memory growth enabled')
    except RuntimeError as e:
        print(e)

In [4]:
# PARAMETERS
data_path = r"C:\Users\ronle\Desktop\BUAS\Y2C\datasets\train_feature_combined_emotion_dataset.csv"
glove_path = r"C:\Users\ronle\Desktop\BUAS\Y2C\glove.6B.300d.txt"
max_words = 10000
max_len = 100
embedding_dim = 300

In [5]:
# LOAD DATA
df = pd.read_csv(data_path)

# Create target label by taking argmax across emotion columns
emotion_cols = ['happiness', 'sadness', 'disgust', 'anger', 'fear', 'surprise', 'neutral']
df['label'] = df[emotion_cols].values.argmax(axis=1)
num_classes = len(emotion_cols)
y = to_categorical(df['label'], num_classes=num_classes)

In [5]:
# Prepare sentiment features and normalize
sentiment_features = df[['TextBlob_Polarity', 'VADER_Compound']].values
scaler = StandardScaler()
sentiment_features = scaler.fit_transform(sentiment_features)

# TOKENIZE TEXT
df['text'] = df['text'].astype(str)
tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(df['text'])
sequences = tokenizer.texts_to_sequences(df['text'])
X_text = pad_sequences(sequences, maxlen=max_len)

In [6]:
# SPLIT DATA (70% train, 20% val, 10% test)
X_text_temp, X_text_test, X_sent_temp, X_sent_test, y_temp, y_test = train_test_split(
    X_text, sentiment_features, y, test_size=0.10, random_state=42, stratify=y)
X_text_train, X_text_val, X_sent_train, X_sent_val, y_train, y_val = train_test_split(
    X_text_temp, X_sent_temp, y_temp, test_size=0.2222, random_state=42, stratify=y_temp)

In [ ]:
# LOAD PRETRAINED GLOVE EMBEDDINGS
embeddings_index = {}
with open(glove_path, encoding='utf8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        coeffs = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = coeffs
print(f'Found {len(embeddings_index)} word vectors in GloVe.')

In [8]:
# CREATE EMBEDDING MATRIX
word_index = tokenizer.word_index
num_words = min(max_words, len(word_index) + 1)
embedding_matrix = np.zeros((num_words, embedding_dim))
for word, i in word_index.items():
    if i >= max_words:
        continue
    embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector

In [9]:
# Define the model architecture
# Text branch
text_input = Input(shape=(max_len,), name='text_input')
embedding_layer = Embedding(input_dim=num_words,
                            output_dim=embedding_dim,
                            weights=[embedding_matrix],
                            input_length=max_len,
                            trainable=False)(text_input)

In [ ]:
# Here we replace LSTM with SimpleRNN
rnn_out = Bidirectional(SimpleRNN(128, dropout=0.2, recurrent_dropout=0.2))(embedding_layer)

# Sentiment branch
sent_input = Input(shape=(sentiment_features.shape[1],), name='sentiment_input')
sent_dense = Dense(32, activation='relu')(sent_input)
sent_dense = Dropout(0.3)(sent_dense)

# Combine branches
combined = Concatenate()([rnn_out, sent_dense])
combined_dense = Dense(64, activation='relu')(combined)
combined_dense = Dropout(0.5)(combined_dense)
output = Dense(num_classes, activation='softmax')(combined_dense)

# Build and compile the RNN model
rnn_model = Model(inputs=[text_input, sent_input], outputs=output)
rnn_model.compile(optimizer=Adam(learning_rate=0.001),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
rnn_model.summary()

In [11]:
# DEFINE CALLBACKS
callbacks = [
    EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True),
    ModelCheckpoint(filepath='best_rnn_model.h5', monitor='val_loss', save_best_only=True)
]

# MODEL DEFINITION: Simple RNN version
# Text branch
text_input = Input(shape=(max_len,), name='text_input')


In [ ]:
# TRAIN THE RNN MODEL
history_rnn = rnn_model.fit(
    {'text_input': X_text_train, 'sentiment_input': X_sent_train},
    y_train,
    epochs=50,
    batch_size=32,
    validation_data=({'text_input': X_text_val, 'sentiment_input': X_sent_val}, y_val),
    callbacks=callbacks
)

In [ ]:
# EVALUATE ON TEST SET
loss_rnn, acc_rnn = rnn_model.evaluate({'text_input': X_text_test, 'sentiment_input': X_sent_test}, y_test)
print(f'RNN Test Loss: {loss_rnn:.4f} / Test Accuracy: {acc_rnn:.4f}')

In [ ]:
# Calculate overall metrics from the classification report
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Get predictions and true labels
y_pred_rnn = rnn_model.predict({'text_input': X_text_test, 'sentiment_input': X_sent_test})
y_pred_rnn_classes = np.argmax(y_pred_rnn, axis=1)
y_true = np.argmax(y_test, axis=1)

# Calculate metrics
accuracy = accuracy_score(y_true, y_pred_rnn_classes)
precision = precision_score(y_true, y_pred_rnn_classes, average='weighted')
recall = recall_score(y_true, y_pred_rnn_classes, average='weighted')
f1 = f1_score(y_true, y_pred_rnn_classes, average='weighted')

# Print overall metrics
print("\nOverall Metrics:")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

In [ ]:
# 1. Learning Curves
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history_rnn.history['loss'], label='Train Loss')
plt.plot(history_rnn.history['val_loss'], label='Validation Loss')
plt.title('RNN Loss Curves')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_rnn.history['accuracy'], label='Train Accuracy')
plt.plot(history_rnn.history['val_accuracy'], label='Validation Accuracy')
plt.title('RNN Accuracy Curves')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

# 2. Confusion Matrix
# Get predictions from the RNN model
y_pred_rnn = rnn_model.predict({'text_input': X_text_test, 'sentiment_input': X_sent_test})
y_pred_rnn_classes = np.argmax(y_pred_rnn, axis=1)
y_true = np.argmax(y_test, axis=1)

cm = confusion_matrix(y_true, y_pred_rnn_classes)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=emotion_cols, yticklabels=emotion_cols)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('RNN Confusion Matrix')
plt.show()

# 3. Classification Report
print("RNN Classification Report:")
print(classification_report(y_true, y_pred_rnn_classes, target_names=emotion_cols))


In [25]:
rnn_model.save('RNN_3.h5')

___

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, Dense, Dropout, Concatenate, Bidirectional
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


In [2]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
print("TensorFlow version:", tf.__version__)
print("GPU Device:", tf.test.gpu_device_name())

Num GPUs Available:  1
TensorFlow version: 2.10.1
GPU Device: /device:GPU:0


In [3]:
data_path = r"C:\Users\ronle\Desktop\BUAS\Y2C\datasets\train_feature_combined_emotion_dataset.csv"
test_dataset_path = r"C:\Users\ronle\Desktop\BUAS\Y2C\datasets\test_dataset.csv"
glove_path = r"C:\Users\ronle\Desktop\BUAS\Y2C\glove.6B.300d.txt"
max_words = 10000
max_len = 100
embedding_dim = 300

In [4]:
df = pd.read_csv(data_path)


In [5]:
# Create target label by taking argmax across emotion columns
emotion_cols = ['happiness', 'sadness', 'disgust', 'anger', 'fear', 'surprise', 'neutral']
df['label'] = df[emotion_cols].values.argmax(axis=1)
num_classes = len(emotion_cols)
y = to_categorical(df['label'], num_classes=num_classes)

In [6]:
# Prepare sentiment features and normalize
sentiment_features = df[['TextBlob_Polarity', 'VADER_Compound']].values
scaler = StandardScaler()
sentiment_features = scaler.fit_transform(sentiment_features)

In [7]:
# TOKENIZE TEXT
df['text'] = df['text'].astype(str)
tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(df['text'])
sequences = tokenizer.texts_to_sequences(df['text'])
X_text = pad_sequences(sequences, maxlen=max_len)

In [8]:
# SPLIT DATA (80% train, 20% val) - We'll use external test set
X_text_train, X_text_val, X_sent_train, X_sent_val, y_train, y_val = train_test_split(
    X_text, sentiment_features, y, test_size=0.20, random_state=42, stratify=y)

In [9]:
# LOAD PRETRAINED GLOVE EMBEDDINGS
embeddings_index = {}
with open(glove_path, encoding='utf8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        coeffs = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = coeffs
print(f'Found {len(embeddings_index)} word vectors in GloVe.')

Found 400000 word vectors in GloVe.


In [10]:
# CREATE EMBEDDING MATRIX
word_index = tokenizer.word_index
num_words = min(max_words, len(word_index) + 1)
embedding_matrix = np.zeros((num_words, embedding_dim))
for word, i in word_index.items():
    if i >= max_words:
        continue
    embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector

In [11]:
# DEFINE MODEL ARCHITECTURE
# Text branch
text_input = Input(shape=(max_len,), name='text_input')
embedding_layer = Embedding(input_dim=num_words,
                           output_dim=embedding_dim,
                           weights=[embedding_matrix],
                           input_length=max_len,
                           trainable=False)(text_input)

# Here we use Bidirectional SimpleRNN
rnn_out = Bidirectional(SimpleRNN(128, dropout=0.2, recurrent_dropout=0.2))(embedding_layer)
text_features = Dense(128, activation='relu')(rnn_out)
text_features = Dropout(0.3)(text_features)

# Sentiment branch
sent_input = Input(shape=(sentiment_features.shape[1],), name='sentiment_input')
sent_dense = Dense(32, activation='relu')(sent_input)
sent_dense = Dropout(0.3)(sent_dense)

# Combine branches
combined = Concatenate()([text_features, sent_dense])
combined_dense = Dense(64, activation='relu')(combined)
combined_dense = Dropout(0.5)(combined_dense)
output = Dense(num_classes, activation='softmax')(combined_dense)

# Build and compile the full model (with sentiment features)
rnn_model = Model(inputs=[text_input, sent_input], outputs=output)
rnn_model.compile(optimizer=Adam(learning_rate=0.001),
                 loss='categorical_crossentropy',
                 metrics=['accuracy'])

rnn_model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 text_input (InputLayer)        [(None, 100)]        0           []                               
                                                                                                  
 embedding (Embedding)          (None, 100, 300)     3000000     ['text_input[0][0]']             
                                                                                                  
 bidirectional (Bidirectional)  (None, 256)          109824      ['embedding[0][0]']              
                                                                                                  
 sentiment_input (InputLayer)   [(None, 2)]          0           []                               
                                                                                              

In [12]:

# DEFINE CALLBACKS
callbacks = [
    EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True),
    ModelCheckpoint(filepath='best_rnn_model.h5', monitor='val_loss', save_best_only=True)
]

In [13]:
# TRAIN THE RNN MODEL
history_rnn = rnn_model.fit(
    {'text_input': X_text_train, 'sentiment_input': X_sent_train},
    y_train,
    epochs=50,
    batch_size=32,
    validation_data=({'text_input': X_text_val, 'sentiment_input': X_sent_val}, y_val),
    callbacks=callbacks
)

Epoch 1/50
15309/15309 [==============================] - 3658s 239ms/step - loss: 1.3603 - accuracy: 0.5188 - val_loss: 1.2715 - val_accuracy: 0.5395
Epoch 2/50
15309/15309 [==============================] - 4051s 265ms/step - loss: 1.2793 - accuracy: 0.5350 - val_loss: 1.2085 - val_accuracy: 0.5525
Epoch 3/50
15309/15309 [==============================] - 4341s 284ms/step - loss: 1.2273 - accuracy: 0.5476 - val_loss: 1.1945 - val_accuracy: 0.5537
Epoch 4/50
15309/15309 [==============================] - 4825s 315ms/step - loss: 1.2274 - accuracy: 0.5468 - val_loss: 1.1934 - val_accuracy: 0.5552
Epoch 5/50
15309/15309 [==============================] - 4804s 314ms/step - loss: 1.2312 - accuracy: 0.5465 - val_loss: 1.1980 - val_accuracy: 0.5509
Epoch 6/50
15309/15309 [==============================] - 4806s 314ms/step - loss: 1.2215 - accuracy: 0.5485 - val_loss: 1.1984 - val_accuracy: 0.5531
Epoch 7/50
15309/15309 [==============================] - 4811s 314ms/step - loss: 1.2200 - ac

In [14]:
# Save the full model
rnn_model.save('RNN_3.h5')

In [15]:

# Create a new model that only uses text input
text_only_input = Input(shape=(max_len,), name='text_input')
# Reuse the embedding layer weights
embedding = Embedding(input_dim=num_words,
                     output_dim=embedding_dim,
                     weights=[embedding_matrix],
                     input_length=max_len,
                     trainable=False)(text_only_input)
# Reuse the RNN layer with trained weights
rnn_layer = Bidirectional(SimpleRNN(128, dropout=0.2, recurrent_dropout=0.2))
rnn_layer.set_weights(rnn_model.get_layer('bidirectional')(rnn_model.get_layer('embedding')(text_input)).get_weights())
rnn_output = rnn_layer(embedding)

# Reuse the text feature dense layer
text_feature_layer = Dense(128, activation='relu')
text_feature_layer.set_weights(rnn_model.get_layer('dense')(rnn_out).get_weights())
text_features_output = text_feature_layer(rnn_output)
text_features_output = Dropout(0.3)(text_features_output)

# Create a new dense layer to replace the combined path
new_dense = Dense(64, activation='relu')(text_features_output)
new_dense = Dropout(0.5)(new_dense)

# Final output layer
final_output_layer = Dense(num_classes, activation='softmax')
final_output_layer.set_weights(rnn_model.get_layer('dense_3')(combined_dense).get_weights())
final_output = final_output_layer(new_dense)

# Create and compile the text-only model
text_only_model = Model(inputs=text_only_input, outputs=final_output)
text_only_model.compile(optimizer=Adam(learning_rate=0.001),
                       loss='categorical_crossentropy',
                       metrics=['accuracy'])

text_only_model.summary()

AttributeError: 'KerasTensor' object has no attribute 'get_weights'

In [ ]:
text_only_model.save('RNN_3v1.h5')


In [ ]:
test_df = pd.read_csv(test_dataset_path)


In [ ]:
# Create target labels for test data
test_df['label'] = test_df[emotion_cols].values.argmax(axis=1)
y_test = to_categorical(test_df['label'], num_classes=num_classes)

In [ ]:
# Tokenize and pad the test text data
test_df['text'] = test_df['text'].astype(str)
test_sequences = tokenizer.texts_to_sequences(test_df['text'])
X_text_test = pad_sequences(test_sequences, maxlen=max_len)

In [ ]:
# EVALUATE ON TEST SET USING TEXT-ONLY MODEL
loss_rnn, acc_rnn = text_only_model.evaluate(X_text_test, y_test)
print(f'RNN Test Loss: {loss_rnn:.4f} / Test Accuracy: {acc_rnn:.4f}')

In [ ]:
# Get predictions and true labels
y_pred_rnn = text_only_model.predict(X_text_test)
y_pred_rnn_classes = np.argmax(y_pred_rnn, axis=1)
y_true = np.argmax(y_test, axis=1)

In [ ]:
# Calculate metrics
accuracy = accuracy_score(y_true, y_pred_rnn_classes)
precision = precision_score(y_true, y_pred_rnn_classes, average='weighted')
recall = recall_score(y_true, y_pred_rnn_classes, average='weighted')
f1 = f1_score(y_true, y_pred_rnn_classes, average='weighted')

In [ ]:
# Print overall metrics
print("\nOverall Metrics:")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

In [ ]:
# 1. Learning Curves
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history_rnn.history['loss'], label='Train Loss')
plt.plot(history_rnn.history['val_loss'], label='Validation Loss')
plt.title('RNN Loss Curves')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_rnn.history['accuracy'], label='Train Accuracy')
plt.plot(history_rnn.history['val_accuracy'], label='Validation Accuracy')
plt.title('RNN Accuracy Curves')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

In [ ]:
# 2. Confusion Matrix
cm = confusion_matrix(y_true, y_pred_rnn_classes)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
           xticklabels=emotion_cols, yticklabels=emotion_cols)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('RNN Confusion Matrix')
plt.show()

In [ ]:
# 3. Classification Report
print("RNN Classification Report:")
print(classification_report(y_true, y_pred_rnn_classes, target_names=emotion_cols))